# GenAi

In [1]:
import os
import openai
from langchain_openai import ChatOpenAI

In [2]:
llm = ChatOpenAI(model="gpt-4o-mini",temperature=0.0)

In [3]:
response1 = llm.invoke("When was Virat Kohli born and how old is he today?")

print("User: When was Virat Kohli born and how old is he today?")
print("LLM:", response1.content)

User: When was Virat Kohli born and how old is he today?
LLM: Virat Kohli was born on November 5, 1988. As of today, October 4, 2023, he is 34 years old. He will turn 35 in about a month.


# AgenticAI _flow

"""

 Pattern : Reason → Pick tool → Execute → Observe → Repeat → Final Answer
 API     : langgraph.prebuilt.create_react_agent
 Packages: langchain==0.3.19  langgraph==0.3.5
           langchain-openai   openai==1.64.0   wikipedia

 Key idea
 --------
 The agent can call REAL tools and loops until the task is done.
 Each tool result is fed back as an Observation, which shapes
 the next Thought.  The agent decides:
   • Which tool to call
   • What input to pass it
   • When enough information has been gathered

 Tools provided here:
   🔢  calculator      — safe eval of math expressions
   📖  wikipedia_search — live Wikipedia summaries
   📅  get_today_date  — returns today's date

 Flow per turn:
     User question
         │
         ▼
     LLM: Thought + tool_call decision
         │
         ▼
     Tool executes  →  returns Observation
         │
         ▼
     LLM receives Observation  →  new Thought
         │  (loop repeats until LLM emits a plain AIMessage)
         ▼
     Final Answer printed
============================================================
"""

In [6]:

import os
import math
import datetime
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
#from langgraph.prebuilt import create_react_agent
from langchain.agents import create_agent


In [7]:

# ══════════════════════════════════════════════════════════════════════════════
#  TOOL DEFINITIONS
#  Each function decorated with @tool becomes a callable the agent can invoke.
#  The docstring is what the LLM reads to decide when to use the tool.
# ══════════════════════════════════════════════════════════════════════════════

@tool
def calculator(expression: str) -> str:
    """
    Evaluate a mathematical expression and return the result as a string.
    Examples of valid inputs: '2025 - 1988', 'sqrt(144)', '365 * 24 * 60', '2 ** 10'.
    Use this whenever the question involves any arithmetic or calculation.
    """
    safe_globals = {
        "__builtins__": {},   # block all built-ins for safety
        "sqrt"  : math.sqrt,
        "pow"   : math.pow,
        "abs"   : abs,
        "round" : round,
        "pi"    : math.pi,
        "e"     : math.e,
        "log"   : math.log,
        "floor" : math.floor,
        "ceil"  : math.ceil,
    }
    try:
        result = eval(expression, safe_globals)   # restricted eval
        return str(result)
    except Exception as ex:
        return f"Calculation error: {ex}"


@tool
def wikipedia_search(query: str) -> str:
    """
    Search Wikipedia and return a short summary (3 sentences) for the query.
    Use this to look up facts about people, places, events, or concepts.
    Example inputs: 'Virat Kohli', 'Python programming language', 'Eiffel Tower'.
    """
    try:
        import wikipedia
        return wikipedia.summary(query, sentences=3, auto_suggest=False)
    except Exception as ex_outer:
        try:
            # Handle disambiguation by picking the first option
            import wikipedia
            options = wikipedia.search(query)
            if options:
                return wikipedia.summary(options[0], sentences=3)
            return f"No Wikipedia results for '{query}'"
        except Exception as ex_inner:
            return f"Wikipedia error: {ex_inner}"


@tool
def get_today_date(unused: str = "") -> str:
    """
    Return today's date as a human-readable string (e.g. '05 November 2025').
    Call this whenever the question involves today's date or the current year.
    The input argument is ignored — pass an empty string.
    """
    return datetime.date.today().strftime("%d %B %Y")

# @tool
# def get_current_year() -> str:
#     """
#     Returns the current year.
#     Use this whenever current year is needed.
#     """
#     return str(datetime.date.today().year)


# ══════════════════════════════════════════════════════════════════════════════
#  HELPERS
# ══════════════════════════════════════════════════════════════════════════════

TOOLS = [calculator, wikipedia_search, get_today_date]

TOOL_ICONS = {
    "calculator"      : "🔢",
    "wikipedia_search": "📖",
    "get_today_date"  : "📅",
}

In [13]:
def print_banner(title: str) -> None:
    width = 60
    print("\n" + "=" * width)
    print(f"  {title}")
    print("=" * width)

In [9]:
def print_message_trace(messages: list) -> None:
    """Pretty-print the full agent message thread step by step."""
    print("\n  ── Agent execution trace ──────────────────────────────")
    step = 0
    for msg in messages:
        kind = type(msg).__name__

        if kind == "HumanMessage":
            print(f"\n  👤 USER\n     {msg.content}")

        elif kind == "AIMessage":
            step += 1
            # The LLM may emit text AND/OR tool_call requests
            if msg.content:
                print(f"\n  🤖 LLM (turn {step})\n     {msg.content}")
            if hasattr(msg, "tool_calls") and msg.tool_calls:
                for tc in msg.tool_calls:
                    icon = TOOL_ICONS.get(tc["name"], "🔧")
                    print(f"\n  {icon} TOOL CALL  →  {tc['name']}")
                    print(f"     Input : {tc['args']}")

        elif kind == "ToolMessage":
            icon = TOOL_ICONS.get(getattr(msg, "name", ""), "🔧")
            print(f"     Result: {msg.content}")

    print("\n  ── End of trace ────────────────────────────────────────")

In [10]:

# ══════════════════════════════════════════════════════════════════════════════
#  MAIN AGENT FUNCTION
# ══════════════════════════════════════════════════════════════════════════════

def agent_with_tools(question: str) -> str:
    """
    Truly agentic: the LLM picks tools, calls them, reads results,
    and loops until it can give a grounded final answer.
    """

    # ── 1. Build the LLM ──────────────────────────────────────────────────────
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

    # ── 2. Create the agent graph  ──────────
    #       No prompt needed — the prebuilt agent injects tool descriptions
    #       automatically from each tool's docstring.
    
    
    agent = create_agent(llm, TOOLS,system_prompt="""
    You must never assume the current year.

    If the current year is needed:
    1. Call get_today_date first.
    2. Read the result.
    3. Extract the year.
    4. Then perform calculations.

    Never perform calculations using an assumed year.
    """)
    

    # ── 3. Run ─────────────────────────────────────────────────────────────────
    print("\n  [→] Agent running (may call multiple tools) …")
    
    result = agent.invoke({
        "messages": [HumanMessage(content=question)]
    })

    # ── 4. Print the full trace ────────────────────────────────────────────────
    messages = result["messages"]
    
    print_message_trace(messages)

    # ── 5. Count stats ─────────────────────────────────────────────────────────
    tool_calls  = sum(1 for m in messages if isinstance(m, ToolMessage))
    llm_turns   = sum(1 for m in messages if isinstance(m, AIMessage))
    print(f"\n  [i] LLM turns      : {llm_turns}")
    print(f"  [i] Tool calls made: {tool_calls}")
    print(f"  [i] Tools available: {len(TOOLS)}")

    # ── 6. Final answer is the last message ───────────────────────────────────
    return messages[-1].content


# ── Entry point ────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    print_banner("Agent WITH Tools")

    questions = [
        # # Needs wikipedia + calculator (two tools, two loop iterations)
        # "When was Virat Kohli born and how old is he today?",

        # # Needs calculator only
        # "What is the square root of 1764?",

#         # Needs wikipedia only
#         "Who is the founder of LangChain?",

        # Needs date tool + calculator (two tools)
        "What year is it, and how many minutes are in that many years since 2000?",
    ]

    for q in questions:
        print(f"\n{'─' * 60}")
        print(f"  Question: {q}")
        answer = agent_with_tools(q)
        print(f"\n  ✅ Final Answer:\n  {answer}")

    print("\n" + "=" * 60)
    print("  What happened:")
    print("  • Agent looped: Thought → Tool call → Observation → Thought")
    print("  • Each tool result was fed back before answering")
    print("  • Maths is NOW correct  (calculator tool was used)")
    print("  • Facts are grounded   (Wikipedia tool was used)")
    print("  • This is Agentic AI — the LLM is a decision-maker,")
    print("    not just a text predictor.")
    print("=" * 60)


  Agent WITH Tools

────────────────────────────────────────────────────────────
  Question: What year is it, and how many minutes are in that many years since 2000?

  [→] Agent running (may call multiple tools) …

  ── Agent execution trace ──────────────────────────────

  👤 USER
     What year is it, and how many minutes are in that many years since 2000?

  📅 TOOL CALL  →  get_today_date
     Input : {}
     Result: 19 June 2026

  🔢 TOOL CALL  →  calculator
     Input : {'expression': '(2026 - 2000) * 365 * 24 * 60'}
     Result: 13665600

  🤖 LLM (turn 3)
     The current year is 2026. Since the year 2000, there have been 13,665,600 minutes.

  ── End of trace ────────────────────────────────────────

  [i] LLM turns      : 3
  [i] Tool calls made: 2
  [i] Tools available: 3

  ✅ Final Answer:
  The current year is 2026. Since the year 2000, there have been 13,665,600 minutes.

  What happened:
  • Agent looped: Thought → Tool call → Observation → Thought
  • Each tool result w

In [11]:
24*60*26*365

13665600

In [12]:
,system_prompt="""
    You must never assume the current year.

    If the current year is needed:
    1. Call get_today_date first.
    2. Read the result.
    3. Extract the year.
    4. Then perform calculations.

    Never perform calculations using an assumed year.
    """